-------------------------------
-------------------------------
# Laboratorio #9 - IA (CC3085)
* Dulce Ambrosio - 231143
* Daniel Chet - 231177
* Gadiel Ocaña - 231270

-------------------------------
-------------------------------

-------------------------------
**Task 2.1:** Generador de Distribución Conjunta

-------------------------------

In [5]:
# ============================================================
# PARÁMETROS DEL MODELO
# ============================================================
# Slide 16 (Ejemplo: El Alarma): Variables B (Robo), E (Terremoto), A (Alarma).
# Slide 18 (CPTs): Priors/Tablas locales para B y E, y la definición local de A dado B,E.
#
# Epsilon (ε): probabilidad base de un evento raro en los priors del ejemplo (Slide 18).
# En este laboratorio se usa ε=0.01 para P(B=1) y P(E=1).

EPSILON = 0.01  # ε = 0.01 según el laboratorio (Slide 18: CPTs/prior de B y E)

print("=" * 50)
print("PARÁMETROS DEL MODELO")
print("=" * 50)
print(f"P(B=1) = ε = {EPSILON}       → probabilidad de robo")
print(f"P(B=0) = 1-ε = {1-EPSILON}   → probabilidad de NO robo")
print(f"P(E=1) = ε = {EPSILON}       → probabilidad de terremoto")
print(f"P(E=0) = 1-ε = {1-EPSILON}   → probabilidad de NO terremoto")
print()
print("P(A=1|B,E): Alarma suena si y solo si B=1 OR E=1 (determinista)")  # Slide 18 (CPT de A|B,E)

PARÁMETROS DEL MODELO
P(B=1) = ε = 0.01       → probabilidad de robo
P(B=0) = 1-ε = 0.99   → probabilidad de NO robo
P(E=1) = ε = 0.01       → probabilidad de terremoto
P(E=0) = 1-ε = 0.99   → probabilidad de NO terremoto

P(A=1|B,E): Alarma suena si y solo si B=1 OR E=1 (determinista)


In [6]:
# ============================================================
# TASK 2.1 — DISTRIBUCIÓN CONJUNTA
# ============================================================
# Slide 15 (Definición de Red Bayesiana) + Slide 17 (Semántica):
#   Una Red Bayesiana define la conjunta como producto de probabilidades locales:
#   P(B,E,A) = P(B) * P(E) * P(A | B,E) (aquí A tiene como padres a B y E).
# Slide 18 (CPTs): usamos los priors P(B) y P(E), y la CPT/local de A|B,E del ejemplo.

def p_b(b):
    """
    P(B=b): Probabilidad prior del Robo.
    Slide 18: CPT/prior de B en el ejemplo de la alarma.
    """
    return EPSILON if b == 1 else (1 - EPSILON)


def p_e(e):
    """
    P(E=e): Probabilidad prior del Terremoto.
    Slide 18: CPT/prior de E en el ejemplo de la alarma.
    """
    return EPSILON if e == 1 else (1 - EPSILON)


def p_a_dado_be(a, b, e):
    """
    P(A=a | B=b, E=e): Probabilidad condicional local de la Alarma.
    Slide 18: CPT de A condicionado a sus padres B y E (en el ejemplo de la alarma).

    """
    # Slide 18: La idea de definir P(A|B,E) localmente viene de las CPTs.
    alarma_esperada = int(bool(b) or bool(e))
    
    # Si el valor de 'a' coincide con lo esperado → probabilidad = 1
    # Si no coincide → probabilidad = 0  (modelo determinista)
    return 1 if a == alarma_esperada else 0


def prob_conjunta(b, e, a):
    """
    P(B=b, E=e, A=a): Distribución conjunta de las 3 variables.

    Slide 15 (Definición) + Slide 17 (Semántica de la red):
        P(B,E,A) = P(B) · P(E) · P(A | B, E)

    Además, por la estructura del DAG del ejemplo (Slide 16):
    - B y E no tienen arista entre sí → se modelan como independientes a priori,
      por eso la conjunta factoriza con P(B)·P(E).
    - A depende de ambos (B y E) como padres.
    """
    return p_b(b) * p_e(e) * p_a_dado_be(a, b, e)


# ---- Verificación: imprimir la tabla de distribución conjunta completa ----
print("=" * 60)
print("TABLA DE DISTRIBUCIÓN CONJUNTA P(B=b, E=e, A=a)")
print("(Equivalente a la tabla de la presentación con ε=0.01)")
print("=" * 60)
print(f"{'B':>4} {'E':>4} {'A':>4} {'P(B,E,A)':>15}")
print("-" * 35)

total = 0
for b in [0, 1]:
    for e in [0, 1]:
        for a in [0, 1]:
            p = prob_conjunta(b, e, a)
            total += p
            print(f"{b:>4} {e:>4} {a:>4} {p:>15.8f}")

print("-" * 35)
print(f"{'SUMA TOTAL':>24} {total:>15.8f}")
print()
print(" La suma total debe ser 1.0 (distribución de probabilidad válida)")

TABLA DE DISTRIBUCIÓN CONJUNTA P(B=b, E=e, A=a)
(Equivalente a la tabla de la presentación con ε=0.01)
   B    E    A        P(B,E,A)
-----------------------------------
   0    0    0      0.98010000
   0    0    1      0.00000000
   0    1    0      0.00000000
   0    1    1      0.00990000
   1    0    0      0.00000000
   1    0    1      0.00990000
   1    1    0      0.00000000
   1    1    1      0.00010000
-----------------------------------
              SUMA TOTAL      1.00000000

 La suma total debe ser 1.0 (distribución de probabilidad válida)


-------------------------------
**Task 2.2:** Inferencia Marginal

-------------------------------

In [7]:
# ============================================================
# TASK 2.2 — INFERENCIA MARGINAL
# ============================================================
# Slide 7 (Inferencia Probabilística): el objetivo es calcular marginales/condicionales
#   a partir de la distribución conjunta.
# Slide 24 (Resumen de Inferencia): formaliza el patrón Query (Q) + Evidencia (E=e).

def inferencia_marginal(query: dict, evidencia: dict = {}) -> float:
    """
    Calcula P(query | evidencia) usando marginalización.

    Referencias a la presentación:
    - Slide 7: Inferencia probabilística (marginales y condicionales desde la conjunta).
    - Slide 24: Query + Evidencia como estructura de una consulta probabilística.

    Parámetros:
        query    : dict con la variable de interés, ej: {'A': 1}
        evidencia: dict con variables observadas, ej: {'B': 1, 'E': 0}
                   Si está vacío, calcula la probabilidad marginal (prior).

    Proceso (marginalización):
        1. Identifica variables ocultas (no están en query ni en evidencia).
        2. Suma P(B,E,A) sobre combinaciones de variables ocultas (Slide 7).
        3. Si hay evidencia, condiciona con: P(X|Y) = P(X,Y) / P(Y) (Slide 7).
    """
    variables = ['B', 'E', 'A']
    
    # Combinamos query + evidencia → valores fijos (Slide 24: Q y evidencia)
    valores_fijos = {**query, **evidencia}
    
    # Las variables ocultas son las que NO están fijas
    variables_ocultas = [v for v in variables if v not in valores_fijos]
    
    # ----  Calcular el numerador P(query, evidencia) ----
    # Slide 7: marginalizar = sumar sobre variables ocultas
    numerador = 0.0
    
    # Generamos todas las combinaciones posibles de variables ocultas
    # (cada una puede ser 0 o 1)
    num_ocultas = len(variables_ocultas)
    for i in range(2 ** num_ocultas):  # 2^n combinaciones
        # Convertimos el índice i a valores binarios para cada variable oculta
        asignacion_ocultas = {}
        for j, var in enumerate(variables_ocultas):
            # Extraemos el bit j del número i
            asignacion_ocultas[var] = (i >> j) & 1
        
        # Combinamos: fijos + ocultos
        asignacion_completa = {**valores_fijos, **asignacion_ocultas}
        
        # Evaluamos la conjunta con esta asignación completa
        p = prob_conjunta(
            asignacion_completa['B'],
            asignacion_completa['E'],
            asignacion_completa['A']
        )
        numerador += p
    
    # ---- Si hay evidencia, calcular el denominador P(evidencia) ----
    # Slide 7: P(X|Y) = P(X,Y) / P(Y)
    if evidencia:
        denominador = inferencia_marginal(evidencia)  # llamada recursiva sin evidencia
        if denominador == 0:
            return 0.0  # evitar división por cero
        return numerador / denominador
    
    return numerador


# ---- Calcular el Prior de la Alarma P(A=1) ----
print("=" * 55)
print("TASK 2.2 — PRIOR DEL EFECTO: P(A=1)")
print("=" * 55)
print()
print("Calculamos P(A=1) sumando sobre todas las combinaciones")
print("posibles de B y E (variables ocultas):")
print()
print("P(A=1) = Σ_b Σ_e  P(B=b, E=e, A=1)")
print()

# Desglose manual para mayor claridad
print("Desglose por combinación:")
suma = 0
for b in [0, 1]:
    for e in [0, 1]:
        p = prob_conjunta(b, e, 1)
        suma += p
        print(f"  P(B={b}, E={e}, A=1) = {p:.8f}")

print(f"  {'':30} --------")
print(f"  {'P(A=1) =':30} {suma:.8f}")
print()

# Usando la función general
p_a1 = inferencia_marginal({'A': 1})
print(f"Resultado con inferencia_marginal: P(A=1) = {p_a1:.8f}")
print()
print("Interpretación: la alarma suena aproximadamente el")
print(f"{p_a1*100:.4f}% del tiempo (muy raro, pues ε es pequeño).")

TASK 2.2 — PRIOR DEL EFECTO: P(A=1)

Calculamos P(A=1) sumando sobre todas las combinaciones
posibles de B y E (variables ocultas):

P(A=1) = Σ_b Σ_e  P(B=b, E=e, A=1)

Desglose por combinación:
  P(B=0, E=0, A=1) = 0.00000000
  P(B=0, E=1, A=1) = 0.00990000
  P(B=1, E=0, A=1) = 0.00990000
  P(B=1, E=1, A=1) = 0.00010000
                                 --------
  P(A=1) =                       0.01990000

Resultado con inferencia_marginal: P(A=1) = 0.01990000

Interpretación: la alarma suena aproximadamente el
1.9900% del tiempo (muy raro, pues ε es pequeño).


-------------------------------
**Task 2.3:** Demostración del Efecto "Explain Away"

-------------------------------

In [8]:
# ============================================================
# TASK 2.3 — DEMOSTRACIÓN DEL EFECTO EXPLAIN AWAY
# ============================================================
# Slide 21 (Patrones de Razonamiento / V-structure): "Explaining Away".
# Slide 22 (Independencia Condicional): al condicionar en el hijo A, los padres (B y E)
#   pueden volverse dependientes, aunque sean independientes a priori.

print("=" * 60)
print("TASK 2.3 — EFECTO EXPLAIN AWAY")
print("=" * 60)

# ---- Cálculo 1: Diagnóstico Simple P(B=1 | A=1) ----
# Slide 24 (Query + Evidencia): Query = {B=1}, Evidencia = {A=1}.
# Pregunta: "La alarma sonó, ¿cuál es la probabilidad de que sea un robo?"
#
# P(B=1 | A=1) = P(B=1, A=1) / P(A=1)  (condicional; ver Slide 7)
#
# Donde P(B=1, A=1) = Σ_e P(B=1, E=e, A=1)  → marginalizamos sobre E (Slide 7)

p_b1_dado_a1 = inferencia_marginal(query={'B': 1}, evidencia={'A': 1})

print()
print("1. DIAGNÓSTICO SIMPLE")
print("-" * 40)
print("Pregunta: 'La alarma sonó, ¿cuál es la")
print("          probabilidad de que sea un robo?'")
print()
print("Fórmula: P(B=1|A=1) = P(B=1, A=1) / P(A=1)")
print()
print(f"  P(B=1 | A=1) = {p_b1_dado_a1:.4f}")
print()
print("Interpretación: Si la alarma suena, hay aproximadamente")
print(f"un {p_b1_dado_a1*100:.2f}% de probabilidad de que sea por un robo.")
print("(Alta probabilidad porque la alarma casi siempre implica robo o terremoto)")

print()
print()

# ---- Cálculo 2: Efecto Explain Away P(B=1 | A=1, E=1) ----
# Slide 21: si sabemos otra causa (E=1), esa causa puede "explicar" A y reducir B.
# Slide 22: este cambio se debe a independencia condicional (dependencia inducida al condicionar en A).
# Slide 24 (Query + Evidencia): Query = {B=1}, Evidencia = {A=1, E=1}.
# Pregunta: "La alarma sonó, pero hubo un terremoto. ¿Sigue siendo probable un robo?"
#
# P(B=1 | A=1, E=1) = P(B=1, A=1, E=1) / P(A=1, E=1)
#
# Ahora B es la única variable oculta → no hay suma, es directo

p_b1_dado_a1_e1 = inferencia_marginal(query={'B': 1}, evidencia={'A': 1, 'E': 1})

print("2. EFECTO EXPLAIN AWAY")
print("-" * 40)
print("Pregunta: 'La alarma sonó, pero me enteré")
print("          que hubo un terremoto. ¿Cuál es")
print("          ahora la probabilidad de robo?'")
print()
print("Fórmula: P(B=1|A=1,E=1) = P(B=1,A=1,E=1) / P(A=1,E=1)")
print()
print(f"  P(B=1 | A=1, E=1) = {p_b1_dado_a1_e1:.4f}")
print()
print("Interpretación: Al saber que hubo un terremoto (otra causa")
print("que explica la alarma), la probabilidad de robo baja drásticamente.")

print()
print()

# ---- Conclusión: Comparación y verificación del Explain Away ----
print("=" * 60)
print("CONCLUSIÓN — COMPARACIÓN DE RESULTADOS")
print("=" * 60)
print()
print(f"  P(B=1 | A=1)        = {p_b1_dado_a1:.4f}  ← sin saber del terremoto")
print(f"  P(B=1 | A=1, E=1)   = {p_b1_dado_a1_e1:.4f}  ← sabiendo que hubo terremoto")
print()

# Verificación matemática del efecto
if p_b1_dado_a1_e1 < p_b1_dado_a1:
    reduccion = p_b1_dado_a1 - p_b1_dado_a1_e1
    print(f"  ✓ EFECTO EXPLAIN AWAY CONFIRMADO")
    print(f"  La probabilidad de robo se redujo en {reduccion:.4f}")
    print(f"  ({reduccion/p_b1_dado_a1*100:.1f}% de reducción relativa)")
else:
    print("  ✗ No se observó el efecto esperado.")

print()
print("-" * 60)
print("EXPLICACIÓN DEL CONCEPTO:")
print("-" * 60)
print("""
El código demuestra numéricamente el Efecto Explain Away:

1. B (Robo) y E (Terremoto) son variables INDEPENDIENTES entre sí.
   No hay arista entre ellas en el DAG (Slide 16).

2. Sin embargo, al CONDICIONAR sobre su efecto común A (Alarma),
   se crea una dependencia indirecta entre B y E (Slide 22).

3. Si sabemos que la alarma sonó (A=1) Y que hubo terremoto (E=1),
   el terremoto 'explica' por sí solo la alarma → ya no necesitamos
   el robo como explicación → P(B=1|A=1,E=1) baja (Slide 21).

4. Esto valida el patrón V-structure / explaining away de la presentación (Slide 21)
   y la idea de independencia condicional (Slide 22).
""")

TASK 2.3 — EFECTO EXPLAIN AWAY

1. DIAGNÓSTICO SIMPLE
----------------------------------------
Pregunta: 'La alarma sonó, ¿cuál es la
          probabilidad de que sea un robo?'

Fórmula: P(B=1|A=1) = P(B=1, A=1) / P(A=1)

  P(B=1 | A=1) = 0.5025

Interpretación: Si la alarma suena, hay aproximadamente
un 50.25% de probabilidad de que sea por un robo.
(Alta probabilidad porque la alarma casi siempre implica robo o terremoto)


2. EFECTO EXPLAIN AWAY
----------------------------------------
Pregunta: 'La alarma sonó, pero me enteré
          que hubo un terremoto. ¿Cuál es
          ahora la probabilidad de robo?'

Fórmula: P(B=1|A=1,E=1) = P(B=1,A=1,E=1) / P(A=1,E=1)

  P(B=1 | A=1, E=1) = 0.0100

Interpretación: Al saber que hubo un terremoto (otra causa
que explica la alarma), la probabilidad de robo baja drásticamente.


CONCLUSIÓN — COMPARACIÓN DE RESULTADOS

  P(B=1 | A=1)        = 0.5025  ← sin saber del terremoto
  P(B=1 | A=1, E=1)   = 0.0100  ← sabiendo que hubo terremoto

  ✓